# Part 2: BiLSTM Model
## Author: Divine Ifechukwude

This notebook builds the second model in our comparison, a Bidirectional LSTM trained on the same shared split Jean prepared in Part 1.

Before training anything, I first check something Jean's baseline results raised a concern about: whether any near-duplicate tweets ended up split across train and validation. If they did, any model, including mine and other, could look better than it really is, just because it already saw something very close to a validation example during training.

In [1]:
!git clone -q https://github.com/Ebi-Tech/NLP-and-Language-Technologies.git

In [2]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

## Step 1: Load the shared split

Before checking anything about duplicates, I need to actually load the file Jean shared and see what it really looks like.

In [3]:
REPO = "NLP-and-Language-Technologies"

data = pd.read_csv(f"{REPO}/splits/train_val_test_split.csv")

print(data.columns.tolist())
print(data["split"].value_counts())
data.head()

['Tweet_ID', 'split']
split
train    27755
test      5948
val       5947
Name: count, dtype: int64


,Tweet_ID,split
0,ID_ALSD9FRM,train
1,ID_ZDTORNZS,train
2,ID_IU9FH9IH,train
3,ID_4U7F8NQH,train
4,ID_MF1I76IL,train


The split file only tells which tweets belong to train, validation, or test, by their ID. To actually see the tweet content and check for duplicates, I need to join this with the file that has the real cleaned text, matching rows up by Tweet_ID. This merge is also a good honesty check on its own: if every ID in the split file finds a match in the text file, I will know nothing got lost or mismatched along the way.

In [4]:
text_data = pd.read_csv(f"{REPO}/data/processed/train_cleaned.csv")

print(text_data.columns.tolist())
print(f"Rows in split file: {len(data)}")
print(f"Rows in text file: {len(text_data)}")

merged = data.merge(text_data, on="Tweet_ID", how="left")

print(f"Rows after merge: {len(merged)}")
print(f"Rows with missing text after merge: {merged['tweet_clean'].isna().sum()}")

merged.head()

['Tweet_ID', 'tweet', 'type', 'label', 'label_id', 'tweet_clean']
Rows in split file: 39650
Rows in text file: 39650
Rows after merge: 39650
Rows with missing text after merge: 0


,Tweet_ID,split,tweet,type,label,label_id,tweet_clean
0,ID_ALSD9FRM,train,I was raped and his commander. Nothing happene...,sexual_violence,sexual_violence,4,i was raped and his commander. nothing happene...
1,ID_ZDTORNZS,train,"A few weeks ago I got drunk, caused a scene an...",sexual_violence,sexual_violence,4,"a few weeks ago i got drunk, caused a scene an..."
2,ID_IU9FH9IH,train,True story: the man who raped me was really ni...,sexual_violence,sexual_violence,4,true story: the man who raped me was really ni...
3,ID_4U7F8NQH,train,I believe in him....he raped me when i was 8,sexual_violence,sexual_violence,4,i believe in him....he raped me when i was 8
4,ID_MF1I76IL,train,I told my boyfriend at the time what happened ...,sexual_violence,sexual_violence,4,i told my boyfriend at the time what happened ...


## Step 2: Checking for duplicate tweets across the split

Now that I have the real tweet text merged with the split labels, I can actually check the thing I was worried about. Jean found 345 groups of near-duplicate tweets in the full dataset, and almost all of them agree on their label, which is good. But that's not quite what I need to know. What I need to know is whether any of those duplicate tweets got separated, with one copy sitting in train and another copy sitting in validation or test. If that happened, my model could end up being tested on something it basically already saw, word for word, during training. That would make my results look better than they actually are, without me realizing it.

So I'm checking all three possible overlaps here: train against validation, train against test, and validation against test. I'm comparing the cleaned tweet text after lowercasing and trimming extra spaces, matching what Jean already did when finding duplicates in the first place.

In [5]:
train_texts = set(merged.loc[merged["split"] == "train", "tweet_clean"].str.strip())
val_texts = set(merged.loc[merged["split"] == "val", "tweet_clean"].str.strip())
test_texts = set(merged.loc[merged["split"] == "test", "tweet_clean"].str.strip())

train_val_overlap = train_texts.intersection(val_texts)
train_test_overlap = train_texts.intersection(test_texts)
val_test_overlap = val_texts.intersection(test_texts)

print(f"Duplicate tweets shared between train and val: {len(train_val_overlap)}")
print(f"Duplicate tweets shared between train and test: {len(train_test_overlap)}")
print(f"Duplicate tweets shared between val and test: {len(val_test_overlap)}")

Duplicate tweets shared between train and val: 113
Duplicate tweets shared between train and test: 104
Duplicate tweets shared between val and test: 29


## What this means

The numbers came back well above zero: 113 duplicate tweets shared between train and validation, 104 between train and test, and 29 between validation and test. Some tweets that are supposed to be kept separate for a fair evaluation are actually sitting on both sides of the split, word for word.

This puts the very high baseline accuracy from the earlier baseline notebook in a different light. A model can look artificially strong on validation if it already saw a near-identical tweet during training, and that is exactly the situation here. This does not replace the keyword-matching explanation found earlier, that is still a real, separate reason the baseline scores so high, but this leakage means the validation number cannot be fully trusted either, on top of that.

Since the shared split file is used by every model in this study, I cannot fix it just for my own results without making my numbers incomparable to everyone else's. The proper fix is removing duplicates before the split is created in the first place, then generating a new split from the cleaned data. Until that happens at the source, I am accounting for it directly in my own evaluation instead: alongside my normal validation score, I also compute a second score using only validation tweets that do not have a duplicate sitting in the training set. If the two scores are close, the leakage likely is not affecting my results much. If they differ noticeably, that is real evidence worth reporting honestly rather than hiding.

In [6]:
val_rows = merged[merged["split"] == "val"].copy()
val_rows["is_leaking"] = val_rows["tweet_clean"].str.strip().isin(train_texts)

print(f"Validation tweets that also appear in train: {val_rows['is_leaking'].sum()}")
print(f"Clean validation tweets (safe to trust): {(~val_rows['is_leaking']).sum()}")

val_clean_ids = val_rows.loc[~val_rows["is_leaking"], "Tweet_ID"]
val_clean_ids.to_csv("val_clean_ids.csv", index=False)

Validation tweets that also appear in train: 130
Clean validation tweets (safe to trust): 5817


## What this shows

Out of the 5,947 validation tweets, 130 of them have an identical twin sitting somewhere in the training set. That leaves 5,817 validation tweets I can actually trust as a fair, untouched test of my model, once it's trained.

One thing worth explaining honestly here: earlier I found 113 duplicate tweets shared between train and validation, but this step found 130 validation rows affected, not 113. That's not a mistake, it's because the earlier number counted unique duplicate tweets, while this number counts every validation row touched by one of those duplicates. So a few of those 113 duplicate tweets must appear more than once within the validation set itself, which pushes the row count up to 130 even though the number of distinct duplicated tweets stayed at 113.

Either way, is fairly small, about 2.2% of the validation set. Once the model is trained, I'll compare my normal validation score against the score on just the 5,817 clean rows, to see whether that 2.2% is actually moving the number in any meaningful way.

## Step 4: Preparing the text for the model

A BiLSTM cannot read raw sentences. It needs every tweet turned into a fixed-length list of numbers, where each number stands for one word. This step does three things: builds a vocabulary from the training tweets, turns every tweet into a sequence of numbers using that vocabulary, and makes every sequence the same length by cutting long ones and padding short ones.

The vocabulary is built only from the training set, not from validation or test. If I let words from validation or test leak into the vocabulary, the model would have an unfair hint about words it should never have seen yet, which defeats the whole point of testing on unseen data.

For the fixed length, the EDA notebook already found that tweets rarely go past 68 words, so I am setting the max length at 70. This keeps almost every tweet whole while still giving every sequence the same size.

I am also building class weights here, not later. The dataset is badly imbalanced, with one class making up over 80% of the tweets and the smallest class under half a percent. Without correcting for this, the model could get a high score just by leaning on the majority class most of the time, the same failure the majority baseline already showed in Part 1, my teammate Jean's work. Class weights make mistakes on the rare classes count for more during training, so the model actually has a reason to learn them.

In [7]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.utils.class_weight import compute_class_weight

MAX_WORDS = 20000
MAX_LEN = 70

train_rows = merged[merged["split"] == "train"]
val_rows = merged[merged["split"] == "val"]
test_rows = merged[merged["split"] == "test"]

tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token="<OOV>")
tokenizer.fit_on_texts(train_rows["tweet_clean"])

X_train = pad_sequences(tokenizer.texts_to_sequences(train_rows["tweet_clean"]), maxlen=MAX_LEN, padding="post", truncating="post")
X_val = pad_sequences(tokenizer.texts_to_sequences(val_rows["tweet_clean"]), maxlen=MAX_LEN, padding="post", truncating="post")
X_test = pad_sequences(tokenizer.texts_to_sequences(test_rows["tweet_clean"]), maxlen=MAX_LEN, padding="post", truncating="post")

y_train = train_rows["label_id"].values
y_val = val_rows["label_id"].values
y_test = test_rows["label_id"].values

class_weights_array = compute_class_weight(class_weight="balanced", classes=np.unique(y_train), y=y_train)
class_weights = dict(enumerate(class_weights_array))

print(f"Vocabulary size found: {len(tokenizer.word_index)}")
print(f"Shape of training sequences: {X_train.shape}")
print(f"Shape of validation sequences: {X_val.shape}")
print(f"Shape of test sequences: {X_test.shape}")
print(f"Class weights: {class_weights}")

Vocabulary size found: 34625
Shape of training sequences: (27755, 70)
Shape of validation sequences: (5947, 70)
Shape of test sequences: (5948, 70)
Class weights: {0: np.float64(36.51973684210526), 1: np.float64(12.173245614035087), 2: np.float64(42.053030303030305), 3: np.float64(1.3337337818356558), 4: np.float64(0.24290027567496608)}


## What this shows

The vocabulary found here is 34,625 words, lower than Jean's 37,767 from the EDA notebook. That's expected rather than a problem: Jean counted vocabulary across the full dataset of 39,650 tweets, while this tokenizer only saw the training split, about 70% of that data. Fewer tweets means fewer unique words, which lines up with the drop.

The sequence shapes match the row counts from the split exactly: 27,755 training rows, 5,947 validation rows, 5,948 test rows, each now turned into a fixed 70-number sequence.

The class weights confirm the imbalance directly. The two rarest classes get weights around 36 and 42, meaning a mistake on either of them counts roughly 36 to 42 times more than a mistake on the majority class, which gets a weight under 0.25. This matches the class percentages found back in Part 1: the majority class barely gets penalized for being wrong, while the smallest classes get pushed hard to actually be learned.

## Step 5: Building the model

Here is the actual architecture. An embedding layer turns each word number into a list of numbers that captures some meaning, then a bidirectional LSTM reads the tweet both forward and backward, which matters for language since a word's meaning can depend on what comes after it, not just what came before. A dropout layer follows to reduce overfitting, since the model could otherwise memorize specific tweets rather than learning general patterns. The final dense layer outputs a score for each of the five classes.

The class weights built in the last step get passed in during training, not built into the architecture itself. That way the model architecture stays simple and the imbalance handling stays a separate, visible decision.

In [8]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Embedding, Bidirectional, LSTM, Dropout, Dense
from tensorflow.keras.callbacks import EarlyStopping

EMBED_DIM = 128
NUM_CLASSES = 5

model = Sequential([
    Input(shape=(MAX_LEN,)),
    Embedding(input_dim=MAX_WORDS, output_dim=EMBED_DIM),
    Bidirectional(LSTM(64, return_sequences=False)),
    Dropout(0.3),
    Dense(NUM_CLASSES, activation="softmax")
])

model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()

I0000 00:00:1790663715.809072      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1790663715.812265      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 70, 128)        │     2,560,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 128)            │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 5)              │           645 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,659,461 (10.15 MB)

 Trainable params: 2,659,461 (10.15 MB)

 Non-trainable params: 0 (0.00 B)

## What this shows

My model is built with real shapes and parameter counts at every layer.

A tweet enters as 70 numbers, one per word position. My embedding layer turns each of those into a 128-number vector, so the shape becomes 70 by 128, that's 2,560,000 parameters for looking up a meaning-vector for each of my 20,000 possible words. My bidirectional LSTM reads through all 70 positions and compresses the tweet into a single 128-number summary. Dropout keeps that same 128-number shape, it randomly zeroes out some of those numbers during training so the model can't lean too hard on any one of them. My final dense layer turns that 128-number summary into 5 numbers, one score per class.

My total trainable parameters come to about 2.66 million, with the embedding layer accounting for over 96% of that, since it's the only layer tied to the full vocabulary size.

## Step 6: Training

Training runs for up to 15 epochs, but stops early if the validation loss stops improving for 3 epochs in a row, so the model doesn't keep training past the point where it's actually getting better. The best weights found during training are restored at the end.

The class weights built earlier get passed in here, this is the one place they actually take effect. Without them, the model would likely just predict the majority class most of the time and still look fairly accurate, the exact trap the majority baseline in Part 1 already fell into.

In [9]:
early_stop = EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=15,
    batch_size=64,
    class_weight=class_weights,
    callbacks=[early_stop]
)

Epoch 1/15
434/434 ━━━━━━━━━━━━━━━━━━━━ 11s 13ms/step - accuracy: 0.9514 - loss: 0.5823 - val_accuracy: 0.9830 - val_loss: 0.0781
Epoch 2/15
434/434 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - accuracy: 0.9809 - loss: 0.1268 - val_accuracy: 0.9919 - val_loss: 0.0440
Epoch 3/15
434/434 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - accuracy: 0.9889 - loss: 0.1617 - val_accuracy: 0.9941 - val_loss: 0.0302
Epoch 4/15
434/434 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - accuracy: 0.9953 - loss: 0.0262 - val_accuracy: 0.9956 - val_loss: 0.0253
Epoch 5/15
434/434 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - accuracy: 0.9942 - loss: 0.0232 - val_accuracy: 0.9948 - val_loss: 0.0296
Epoch 6/15
434/434 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - accuracy: 0.9948 - loss: 0.0289 - val_accuracy: 0.9950 - val_loss: 0.0231
Epoch 7/15
434/434 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - accuracy: 0.9968 - loss: 0.0088 - val_accuracy: 0.9956 - val_loss: 0.0199
Epoch 8/15
434/434 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - accuracy: 0.9967 - loss: 0.0055 - val_acc

## What this shows

Training ran the full 15 epochs, early stopping never triggered. The best validation loss was 0.0107 at epoch 14, and epoch 15 came in slightly higher at 0.0122, one epoch without improvement, short of the 3 needed to stop early.

Training accuracy reaches 0.9998 by the last epoch, meaning the model gets almost every training example right. Validation accuracy settles between 0.995 and 0.997 from around epoch 9 onward, ending at 0.9971.

This validation accuracy lands close to the TF-IDF baseline's own 99.70%. The baseline reached that number by relying on near-literal keyword matches, and the shared split has known leakage between train and validation. Neither of those has been checked against this model yet, so the next step is running the same leak-free comparison built earlier against this model's actual predictions, plus a full breakdown beyond accuracy, since accuracy alone isn't the metric the group agreed to base conclusions on.

## Step 7: Evaluating properly

Accuracy alone doesn't tell me enough on a dataset this imbalanced, a model can score high while still failing the rare classes completely, exactly what the majority baseline showed in Part 1. Here I get predictions on the full validation set, then compute macro-F1 and per-class precision, recall, and F1, plus a confusion matrix to see exactly which classes get confused with which.

In [10]:
from sklearn.metrics import classification_report, confusion_matrix, f1_score

val_preds = model.predict(X_val).argmax(axis=1)

macro_f1 = f1_score(y_val, val_preds, average="macro")
print(f"Macro F1 on full validation set: {macro_f1:.4f}")
print()
print(classification_report(y_val, val_preds, target_names=sorted(merged["label"].unique())))
print()
print("Confusion matrix:")
print(confusion_matrix(y_val, val_preds))

186/186 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step
Macro F1 on full validation set: 0.9746

                              precision    recall  f1-score   support

           economic_violence       0.97      0.97      0.97        33
          emotional_violence       1.00      0.99      0.99        97
harmful_traditional_practice       0.87      0.96      0.92        28
           physical_violence       1.00      0.99      0.99       892
             sexual_violence       1.00      1.00      1.00      4897

                    accuracy                           1.00      5947
                   macro avg       0.97      0.98      0.97      5947
                weighted avg       1.00      1.00      1.00      5947


Confusion matrix:
[[  32    0    0    0    1]
 [   0   96    0    0    1]
 [   0    0   27    0    1]
 [   0    0    0  887    5]
 [   1    0    4    4 4888]]


## Step 8: Checking against the leak-free validation set

Earlier I saved the IDs of the 5,817 validation tweets that don't have a duplicate in training. Here I score the model against only those rows, to see whether the leaking 130 rows are inflating the numbers above.

In [11]:
val_clean_ids = pd.read_csv("val_clean_ids.csv")["Tweet_ID"]
val_rows_reset = val_rows.reset_index(drop=True)
clean_mask = val_rows_reset["Tweet_ID"].isin(val_clean_ids).values

macro_f1_clean = f1_score(y_val[clean_mask], val_preds[clean_mask], average="macro")
accuracy_clean = (y_val[clean_mask] == val_preds[clean_mask]).mean()

print(f"Accuracy on leak-free validation rows: {accuracy_clean:.4f}")
print(f"Macro F1 on leak-free validation rows: {macro_f1_clean:.4f}")

Accuracy on leak-free validation rows: 0.9972
Macro F1 on leak-free validation rows: 0.9747


## What this shows

Full validation macro F1 is 0.9746. Leak-free macro F1 is 0.9747, and leak-free accuracy is 0.9972, both nearly identical to the scores on the complete validation set (overall accuracy 0.9971 once calculated from the confusion matrix, 5930 correct out of 5947). Removing the 130 leaking rows changes almost nothing here.

The confusion matrix shows where the remaining errors sit. The clearest case is `harmful_traditional_practice`, precision 0.87, the lowest of the five classes, caused by 4 `sexual_violence` tweets getting misclassified into it. Everywhere else, errors are single digits: 1 `sexual_violence` tweet predicted as `economic_violence`, 1 predicted as `emotional_violence`, 4 `physical_violence` tweets predicted as `sexual_violence`.

Per-class F1 ranges from 0.92 (`harmful_traditional_practice`, the smallest class at 28 validation examples) up to 1.00 (`sexual_violence`, the largest at 4,897). This is the opposite pattern from the majority baseline in Part 1, which scored zero on every class except the majority one.

## Step 9: Trying a second configuration

The first version scored well, but a single run isn't tuning, it's just one attempt accepted as final. Here I try one deliberate change and compare it against the first version on the same validation set, so there's actual evidence behind whichever configuration I keep.

The change I'm testing is the LSTM's hidden size, dropping it from 64 units to 32. The first model reached 0.9998 training accuracy while validation stayed around 0.997, a gap that points toward some overfitting, the model fitting training data more tightly than it needs to. A smaller hidden layer gives the model less room to memorize specific examples, so if validation performance holds steady or improves while that gap shrinks, that's real evidence the smaller model generalizes at least as well with less capacity. If it gets worse, that's useful evidence too, it means the original size was doing well, not just overfitting.

In [12]:
model_v2 = Sequential([
    Input(shape=(MAX_LEN,)),
    Embedding(input_dim=MAX_WORDS, output_dim=EMBED_DIM),
    Bidirectional(LSTM(32, return_sequences=False)),
    Dropout(0.3),
    Dense(NUM_CLASSES, activation="softmax")
])

model_v2.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])

early_stop_v2 = EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)

history_v2 = model_v2.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=15,
    batch_size=64,
    class_weight=class_weights,
    callbacks=[early_stop_v2]
)

Epoch 1/15
434/434 ━━━━━━━━━━━━━━━━━━━━ 8s 13ms/step - accuracy: 0.9200 - loss: 0.8118 - val_accuracy: 0.9837 - val_loss: 0.0908
Epoch 2/15
434/434 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - accuracy: 0.9861 - loss: 0.1681 - val_accuracy: 0.9889 - val_loss: 0.0538
Epoch 3/15
434/434 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - accuracy: 0.9912 - loss: 0.0632 - val_accuracy: 0.9936 - val_loss: 0.0282
Epoch 4/15
434/434 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - accuracy: 0.9904 - loss: 0.0517 - val_accuracy: 0.9923 - val_loss: 0.0367
Epoch 5/15
434/434 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - accuracy: 0.9943 - loss: 0.0399 - val_accuracy: 0.9946 - val_loss: 0.0243
Epoch 6/15
434/434 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - accuracy: 0.9963 - loss: 0.0225 - val_accuracy: 0.9960 - val_loss: 0.0228
Epoch 7/15
434/434 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - accuracy: 0.9963 - loss: 0.0225 - val_accuracy: 0.9919 - val_loss: 0.0436
Epoch 8/15
434/434 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - accuracy: 0.9937 - loss: 0.0207 - val_accu

## What this shows

Training also ran the full 15 epochs without early stopping. Best validation loss was 0.0140 at epoch 14, with epoch 15 close behind at 0.0142, similarly one epoch without improvement.

Final training accuracy is 0.9991, against a final validation accuracy of 0.9975. The gap between them is smaller than the first version's (0.9998 training versus 0.9971 validation), which lines up with a smaller model overfitting less. The training accuracy also moved less smoothly this time, dipping at epochs 4, 7, 10, and 12 before recovering, instead of climbing steadily like the first version did.

Validation accuracy alone doesn't confirm whether this configuration is actually better, since accuracy isn't the metric this comparison should rest on. The next step computes macro-F1 for this version directly, to compare against the first version's 0.9746.

## Step 10: Comparing the two configurations

Accuracy looked close between both versions, but macro-F1 is what actually matters here, since it treats every class equally regardless of size. Here I get this version's predictions and macro-F1 on the same validation set, then put both versions side by side to decide which one to keep.

In [13]:
val_preds_v2 = model_v2.predict(X_val).argmax(axis=1)
macro_f1_v2 = f1_score(y_val, val_preds_v2, average="macro")

comparison = pd.DataFrame({
    "version": ["64 units", "32 units"],
    "val_accuracy": [0.9971, 0.9975],
    "val_macro_f1": [macro_f1, macro_f1_v2]
})
print(comparison)

186/186 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step
    version  val_accuracy  val_macro_f1
0  64 units        0.9971      0.974597
1  32 units        0.9975      0.977990


## What this shows

The 32-unit version wins on both measures, val_accuracy 0.9975 against 0.9971, val_macro_f1 0.9780 against 0.9746. The smaller model is the final BiLSTM configuration.

This also matches what the training curves showed. The 64-unit version reached near-perfect training accuracy (0.9998) while validation stayed lower, the larger gap pointing to more overfitting. The 32-unit version's smaller gap (0.9991 training versus 0.9975 validation) lines up with it generalizing slightly better, not just training more smoothly.

## Step 11: Evaluating on the held-out test set

Everything so far has used validation, for training decisions and comparing configurations. The test set has not been touched until now. This is the one score that gets reported as the model's actual result, evaluated once, with the winning configuration only.

In [14]:
test_preds = model_v2.predict(X_test).argmax(axis=1)

test_macro_f1 = f1_score(y_test, test_preds, average="macro")
test_accuracy = (y_test == test_preds).mean()

print(f"Test accuracy: {test_accuracy:.4f}")
print(f"Test macro F1: {test_macro_f1:.4f}")
print()
print(classification_report(y_test, test_preds, target_names=sorted(merged["label"].unique())))
print()
print("Confusion matrix:")
print(confusion_matrix(y_test, test_preds))

186/186 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step
Test accuracy: 0.9968
Test macro F1: 0.9723

                              precision    recall  f1-score   support

           economic_violence       1.00      0.97      0.98        32
          emotional_violence       0.89      0.98      0.93        98
harmful_traditional_practice       0.90      1.00      0.95        28
           physical_violence       1.00      1.00      1.00       892
             sexual_violence       1.00      1.00      1.00      4898

                    accuracy                           1.00      5948
                   macro avg       0.96      0.99      0.97      5948
                weighted avg       1.00      1.00      1.00      5948


Confusion matrix:
[[  31    1    0    0    0]
 [   0   96    1    1    0]
 [   0    0   28    0    0]
 [   0    0    0  891    1]
 [   0   11    2    2 4883]]


## What this shows

Test accuracy is 0.9968, test macro F1 is 0.9723, close to but slightly below the validation macro F1 of 0.9780. This is the model's actual, final result, evaluated once on data it never influenced any decision with.

The lowest precision is `emotional_violence` at 0.89. The confusion matrix shows why: 11 `sexual_violence` tweets got predicted as `emotional_violence`, the single largest error in the whole matrix. This is a different main error than what showed up on validation, where the largest confusion was between `harmful_traditional_practice` and `sexual_violence`. `harmful_traditional_practice` itself scores perfect recall on test (1.00) but precision only 0.90, from 2 `physical_violence` and 1 `economic_violence` tweet being predicted into it.

`physical_violence` and `sexual_violence`, the two largest classes, both score at or near 1.00 across precision, recall, and F1 on test.

## Step 12: Saving results for the group comparison

Every model in this study needs its results sitting in one shared file so the final comparison notebook can pull from all four parts without anyone re-running anyone else's model. Since I'm the first one adding to this file, I'm defining its structure here: model name, which split the score came from, accuracy, macro F1, per-class F1 for each of the five classes, training time, and hardware used. Everyone else's model appends a row in this same format.

In [15]:
import time

results_row = pd.DataFrame([{
    "model": "BiLSTM",
    "split": "test",
    "accuracy": test_accuracy,
    "macro_f1": test_macro_f1,
    "f1_economic_violence": 0.98,
    "f1_emotional_violence": 0.93,
    "f1_harmful_traditional_practice": 0.95,
    "f1_physical_violence": 1.00,
    "f1_sexual_violence": 1.00,
    "hardware": "GPU (Tesla T4)",
    "notes": "32-unit BiLSTM, selected over 64-unit config on validation macro F1"
}])

import os
results_path = f"{REPO}/results/model_comparison.csv"

if os.path.exists(results_path):
    existing = pd.read_csv(results_path)
    combined = pd.concat([existing, results_row], ignore_index=True)
else:
    os.makedirs(os.path.dirname(results_path), exist_ok=True)
    combined = results_row

combined.to_csv(results_path, index=False)
print(combined)

    model split  accuracy  macro_f1  f1_economic_violence  \
0  BiLSTM  test  0.996806  0.972289                  0.98   

   f1_emotional_violence  f1_harmful_traditional_practice  \
0                   0.93                             0.95   

   f1_physical_violence  f1_sexual_violence        hardware  \
0                   1.0                 1.0  GPU (Tesla T4)   

                                               notes  
0  32-unit BiLSTM, selected over 64-unit config o...  


## Step 13: Related work

I looked into research on each of the five approaches in this study, focusing on work involving short, user-generated text, since that is what this dataset actually is, and work dealing with imbalanced, sensitive classification problems close to this one.

**Classical baseline.** TF-IDF combined with a linear classifier has long been a standard, strong baseline for text classification before deep learning became common, and it remains a fair comparison point precisely because it is simple and has no notion of word order at all. This matters directly for this project: the baseline notebook already found that TF-IDF and Logistic Regression reach 99.70% accuracy here mostly by relying on individual trigger words rather than sentence meaning. That result is itself evidence for why the brief asks for models capable of learning sequential dependencies, a model with no sense of order can still succeed on this dataset if the classes are separable by vocabulary alone, and testing whether the neural models can do better than that, not just match it, is a genuine question this comparison needs to answer.

**BiLSTM.** Recurrent networks, and LSTMs specifically, have been used repeatedly for hate speech and abusive content classification on Twitter, since they read a sentence in order rather than as a bag of words, which lets them pick up on how words relate to each other across a sentence rather than just which words are present. Zhang, Robinson, and Tepper combined a convolutional layer with a gated recurrent network for hate speech detection on Twitter and found that adding a recurrent component that captures word order improved results over convolution alone on most of the datasets they tested [1]. Their result is direct support for choosing a recurrent model here specifically because it captures something a bag-of-words or convolution-only model does not.

**TextCNN.** Kim introduced the use of convolutional filters over word embeddings for sentence classification, showing that a fairly simple CNN, with filters sliding over a few words at a time, performs strongly on sentence-level classification tasks despite having no explicit memory of the whole sequence like an LSTM does [2]. This is the basis for including a TextCNN in this comparison. It is a genuinely different mechanism from the BiLSTM, one looks for short, local word patterns through a fixed-size window, the other reads the whole sequence in order, so comparing them tests two different assumptions about what kind of pattern actually distinguishes these five categories.

**Transformer.** Devlin et al. introduced BERT, which reads a whole sentence at once and lets every word attend to every other word regardless of distance, rather than processing words strictly in order the way an LSTM does [3]. Sanh et al. later introduced DistilBERT, a smaller, faster version of BERT trained to match most of its performance at a fraction of the size and training cost [4], which is the practical reason it is the transformer used in this comparison rather than the full-size model.

**Evaluation under imbalance.** Sokolova and Lapalme's analysis of classification performance measures explains directly why accuracy is a poor choice on an imbalanced dataset like this one: accuracy can stay high even when a model completely fails on minority classes, while macro-averaged measures treat every class equally regardless of its size, which is the more honest reflection of whether a model actually works across all five categories [5]. This is the basis for the metric choice described next.

## Step 14: Justifying the evaluation metrics

Accuracy is reported because it is the actual leaderboard metric for this challenge, so it needs to be on record. But it cannot be the metric decisions are based on here. The majority-class baseline in the earlier notebook already proved this directly: it reaches 82.34% accuracy while scoring zero precision, recall, and F1 on four of the five classes. A high accuracy number on this dataset does not mean a model works, it can just mean the model leans on the largest class.

Macro F1 is used as the primary metric instead. It calculates F1 separately for each of the five classes and then averages those five scores with equal weight, so a model cannot hide a complete failure on `harmful_traditional_practice` (188 examples) behind strong performance on `sexual_violence` (32,647 examples). This matches exactly what Sokolova and Lapalme describe as the purpose of macro-averaging, treating every class as equally important regardless of how many examples it has [5].

Per-class precision, recall, and F1 are reported alongside macro F1, not replaced by it, since a single averaged number can still hide which specific class is struggling and in which direction, whether a model is missing a class entirely (low recall) or wrongly labelling other things as that class (low precision). A confusion matrix is reported for the same reason, to see exactly which classes get confused with which, rather than only knowing that some confusion exists.

## References

[1] Z. Zhang, D. Robinson, and J. Tepper, "Detecting hate speech on Twitter using a convolution-GRU based deep neural network," in *Proc. 15th Extended Semantic Web Conf. (ESWC)*, Heraklion, Greece, 2018, pp. 745-760.

[2] Y. Kim, "Convolutional neural networks for sentence classification," in *Proc. 2014 Conf. Empirical Methods in Natural Language Processing (EMNLP)*, Doha, Qatar, 2014, pp. 1746-1751.

[3] J. Devlin, M. W. Chang, K. Lee, and K. Toutanova, "BERT: Pre-training of deep bidirectional transformers for language understanding," in *Proc. 2019 Conf. North American Chapter Assoc. Comput. Linguistics (NAACL)*, Minneapolis, MN, USA, 2019, pp. 4171-4186.

[4] V. Sanh, L. Debut, J. Chaumond, and T. Wolf, "DistilBERT, a distilled version of BERT: smaller, faster, cheaper and lighter," arXiv:1910.01108, 2019.

[5] M. Sokolova and G. Lapalme, "A systematic analysis of performance measures for classification tasks," *Inf. Process. Manag.*, vol. 45, no. 4, pp. 427-437, 2009.